In [ ]:
from pathlib import Path

pdf_path = Path("../docs/Approved_Document_M_Volume_2.pdf")

print(pdf_path.exists())

In [ ]:
from pypdf import PdfReader

reader = PdfReader("../docs/Approved_Document_M_Volume_2.pdf")

print("Pages:", len(reader.pages))

In [ ]:
text = ""

for page in reader.pages[:20]:
    text += page.extract_text() + "\n"

print(text[:5000])

In [ ]:
full_text = ""

for i, page in enumerate(reader.pages):
    page_text = page.extract_text()

    if page_text:
        full_text += f"\n\n--- PAGE {i+1} ---\n\n"
        full_text += page_text

In [ ]:
section5_start = full_text.find("Section 5: Sanitary accommodation")

print(section5_start)

In [ ]:
section6_start = full_text.find("Standards referred to")

section5_text = full_text[section5_start:section6_start]

print(section5_text[:8000])

In [ ]:
from pathlib import Path

output_path = Path("../outputs/section5_sanitary_accommodation.txt")
output_path.write_text(section5_text, encoding="utf-8")

print("Saved:", output_path)

In [ ]:
for i, page in enumerate(reader.pages):
    page_text = page.extract_text()

    if page_text and "Wheelchair-accessible unisex toilets" in page_text:
        print("Found on page:", i + 1)
        print(page_text[:3000])
        print("=" * 80)

In [ ]:
target_pages_text = ""

for page_number in range(49, 58):  # PDF pages 50 to 58
    page_text = reader.pages[page_number].extract_text()
    if page_text:
        target_pages_text += f"\n\n--- PAGE {page_number + 1} ---\n\n"
        target_pages_text += page_text

print(target_pages_text[:12000])

In [ ]:
target_path = Path("../outputs/partm_accessible_wc_shower_text.txt")
target_path.write_text(target_pages_text, encoding="utf-8")

print("Saved:", target_path)

In [ ]:
import pandas as pd

In [ ]:
partm_rules = [
    {
        "Rule_ID": "PM-WC-01",
        "Requirement": "Wheelchair-accessible unisex toilet provided",
        "Automatable": "Yes",
        "Current_Status": "PASS"
    },
    {
        "Rule_ID": "PM-WC-02",
        "Requirement": "Accessible toilet contains toilet pack",
        "Automatable": "Yes",
        "Current_Status": "PASS"
    },
    {
        "Rule_ID": "PM-WC-03",
        "Requirement": "Accessible toilet contains grab rails",
        "Automatable": "Yes",
        "Current_Status": "PASS"
    },
    {
        "Rule_ID": "PM-WC-04",
        "Requirement": "Accessible toilet door width compliant",
        "Automatable": "Yes",
        "Current_Status": "PASS"
    },
    {
        "Rule_ID": "PM-WC-05",
        "Requirement": "Transfer space beside WC",
        "Automatable": "Potentially",
        "Current_Status": "Not Yet Checked"
    },
    {
        "Rule_ID": "PM-WC-06",
        "Requirement": "Room dimensions comply with Diagram 18",
        "Automatable": "Potentially",
        "Current_Status": "Not Yet Checked"
    }
]

rules_df = pd.DataFrame(partm_rules)

rules_df

In [ ]:
import ifcopenshell

model = ifcopenshell.open("../data/ifc/sample_model.ifc")

spaces = model.by_type("IfcSpace")

print("Spaces loaded:", len(spaces))

In [ ]:
for space in spaces:

    if space.LongName in ["DDA WC", "DDA Shower Room"]:

        print("\nSPACE:", space.Name, "-", space.LongName)

        print("Representation:", space.Representation is not None)

        if hasattr(space, "ObjectPlacement"):
            print("Placement:", space.ObjectPlacement is not None)

In [ ]:
import ifcopenshell.util.element

In [ ]:
for space in spaces:

    if space.LongName in ["DDA WC", "DDA Shower Room"]:

        print("\nSPACE:", space.Name, "-", space.LongName)

        try:
            print(ifcopenshell.util.element.get_psets(space))
        except:
            print("No psets")

In [ ]:
import ifcopenshell.geom

settings = ifcopenshell.geom.settings()
settings.set(settings.USE_WORLD_COORDS, True)

for space in spaces:
    if space.LongName in ["DDA WC", "DDA Shower Room"]:
        print("\nSPACE:", space.Name, "-", space.LongName)

        shape = ifcopenshell.geom.create_shape(settings, space)
        vertices = shape.geometry.verts

        xs = vertices[0::3]
        ys = vertices[1::3]
        zs = vertices[2::3]

        width_x = max(xs) - min(xs)
        width_y = max(ys) - min(ys)
        height_z = max(zs) - min(zs)

        print("Approx X dimension:", round(width_x, 2))
        print("Approx Y dimension:", round(width_y, 2))
        print("Approx height:", round(height_z, 2))

In [ ]:
space_dimension_results = []

for space in spaces:
    if space.LongName in ["DDA WC", "DDA Shower Room"]:

        shape = ifcopenshell.geom.create_shape(settings, space)
        vertices = shape.geometry.verts

        xs = vertices[0::3]
        ys = vertices[1::3]
        zs = vertices[2::3]

        x_dim = max(xs) - min(xs)
        y_dim = max(ys) - min(ys)
        height = max(zs) - min(zs)

        min_dim = min(x_dim, y_dim)

        turning_circle_status = "PASS" if min_dim >= 1.5 else "FAIL"

        space_dimension_results.append({
            "Space": space.LongName,
            "X_Dimension_m": round(x_dim, 2),
            "Y_Dimension_m": round(y_dim, 2),
            "Height_m": round(height, 2),
            "Minimum_Plan_Dimension_m": round(min_dim, 2),
            "Turning_Circle_1500mm_Check": turning_circle_status
        })

space_dimensions_df = pd.DataFrame(space_dimension_results)

space_dimensions_df

In [ ]:
space_dimensions_df.to_excel("../outputs/dda_space_dimension_checks.xlsx", index=False)

print("DDA space dimension checks saved.")

In [ ]:
compliance_summary = [
    {
        "Rule": "Accessible WC detected",
        "Source": "IFC Space",
        "Result": "PASS"
    },
    {
        "Rule": "Accessible Shower detected",
        "Source": "IFC Space",
        "Result": "PASS"
    },
    {
        "Rule": "Accessible door widths compliant",
        "Source": "IFC Door Data",
        "Result": "PASS"
    },
    {
        "Rule": "Doc M toilet/shower packs detected",
        "Source": "IFC Fixtures",
        "Result": "PASS"
    },
    {
        "Rule": "Required accessibility fixtures detected",
        "Source": "IFC Fixtures",
        "Result": "PASS"
    },
    {
        "Rule": "1500mm turning circle check",
        "Source": "IFC Geometry",
        "Result": "PASS"
    },
    {
        "Rule": "Diagram 18 detailed layout verification",
        "Source": "Future Geometry Analysis",
        "Result": "REVIEW"
    }
]

compliance_report_df = pd.DataFrame(compliance_summary)

compliance_report_df

In [ ]:
compliance_report_df.to_excel("../outputs/final_compliance_report.xlsx", index=False)

print("Final compliance report saved.")

In [ ]:
dda_elements = []

for space in spaces:
    if space.LongName in ["DDA WC", "DDA Shower Room"]:
        if hasattr(space, "ContainsElements"):
            for rel in space.ContainsElements:
                for element in rel.RelatedElements:
                    dda_elements.append({
                        "Space_Name": space.Name,
                        "Space_LongName": space.LongName,
                        "IFC_Type": element.is_a(),
                        "Element_Name": element.Name
                    })

dda_elements_df = pd.DataFrame(dda_elements)

dda_elements_df

In [ ]:
def check_keyword_in_space(space_name, keywords):
    elements = dda_elements_df[
        dda_elements_df["Space_LongName"] == space_name
    ]

    combined_text = " ".join(elements["Element_Name"].astype(str)).lower()

    return any(keyword.lower() in combined_text for keyword in keywords)


automated_rules = [
    {
        "Rule_ID": "PM-WC-01",
        "Space": "DDA WC",
        "Requirement": "Doc M toilet pack detected",
        "Check_Function": lambda: check_keyword_in_space("DDA WC", ["docm", "doc m", "toiletpack", "toilet pack"])
    },
    {
        "Rule_ID": "PM-WC-02",
        "Space": "DDA WC",
        "Requirement": "Grab rail detected",
        "Check_Function": lambda: check_keyword_in_space("DDA WC", ["grabbar", "grab bar", "support rail", "suprt"])
    },
    {
        "Rule_ID": "PM-WC-03",
        "Space": "DDA WC",
        "Requirement": "Mirror detected",
        "Check_Function": lambda: check_keyword_in_space("DDA WC", ["mirror"])
    },
    {
        "Rule_ID": "PM-SH-01",
        "Space": "DDA Shower Room",
        "Requirement": "Doc M shower pack detected",
        "Check_Function": lambda: check_keyword_in_space("DDA Shower Room", ["docm", "doc m", "showerpack", "shower pack"])
    },
    {
        "Rule_ID": "PM-SP-01",
        "Space": "DDA WC / DDA Shower Room",
        "Requirement": "1500mm turning circle possible from bounding box",
        "Check_Function": lambda: all(space_dimensions_df["Minimum_Plan_Dimension_m"] >= 1.5)
    }
]

results = []

for rule in automated_rules:
    passed = rule["Check_Function"]()

    results.append({
        "Rule_ID": rule["Rule_ID"],
        "Space": rule["Space"],
        "Requirement": rule["Requirement"],
        "Result": "PASS" if passed else "FAIL"
    })

automated_compliance_df = pd.DataFrame(results)

automated_compliance_df

In [ ]:
automated_compliance_df.to_excel("../outputs/automated_compliance_results.xlsx", index=False)

print("Automated compliance results saved.")

In [ ]:
prompt = f"""
You are assisting with automated building regulation compliance checking.

Task:
Extract machine-readable accessibility compliance rules from the following Approved Document M text.

For each rule, return a JSON object with:
- rule_id
- part_m_reference
- requirement
- space_type
- ifc_entities_needed
- check_type
- automatable
- reason

Use only the provided text. Do not invent requirements.

Classify automatable as:
- yes
- partially
- no

Text:
{target_pages_text}
"""

prompt_path = Path("../outputs/llm_rule_extraction_prompt.txt")
prompt_path.write_text(prompt, encoding="utf-8")

print("Prompt saved:", prompt_path)

In [ ]:
import os
from google import genai

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError("GEMINI_API_KEY environment variable is not set.")

client = genai.Client(api_key=api_key)


In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello"
)

print(response.text)

In [ ]:
wc_start = target_pages_text.find("5.10 Wheelchair-accessible unisex toilets")
wc_end = target_pages_text.find("Toilets in separate-sex washrooms")

wc_text = target_pages_text[wc_start:wc_end]

print("WC text length:", len(wc_text))
print(wc_text[:3000])

In [ ]:
shower_start = target_pages_text.find("5.18 Wheelchair-accessible changing and")
shower_end = target_pages_text.find("Wheelchair-accessible bathrooms")

shower_text = target_pages_text[shower_start:shower_end]

print("Shower text length:", len(shower_text))
print(shower_text[:3000])

In [ ]:
focused_partm_text = wc_text + "\n\n" + shower_text

print("Focused text length:", len(focused_partm_text))

In [ ]:
prompt = f"""
You are an expert in UK Building Regulations Part M and BIM-based compliance checking.

Extract machine-readable accessibility compliance rules from the text below.

Return ONLY a valid JSON array.
No markdown. No explanation.

Each rule must contain:
{{
  "rule_id": "",
  "part_m_reference": "",
  "requirement": "",
  "space_type": "",
  "ifc_entities_needed": [],
  "check_type": "",
  "automatable": "",
  "reason": ""
}}

Only use the provided text.
Focus on rules that could support checking wheelchair-accessible unisex toilets and wheelchair-accessible shower facilities.

Text:
{focused_partm_text}
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

raw_output = response.text
print(raw_output[:3000])

In [ ]:
import json

In [ ]:
cleaned = raw_output.strip()

if cleaned.startswith("```json"):
    cleaned = cleaned.replace("```json", "").replace("```", "").strip()
elif cleaned.startswith("```"):
    cleaned = cleaned.replace("```", "").strip()

focused_rules = json.loads(cleaned)

focused_llm_rules_df = pd.DataFrame(focused_rules)

focused_llm_rules_df

In [ ]:
focused_llm_rules_df.to_excel("../outputs/focused_llm_partm_rules_wc_shower.xlsx", index=False)

Path("../outputs/focused_llm_partm_rules_wc_shower.json").write_text(
    json.dumps(focused_rules, indent=2),
    encoding="utf-8"
)

print("Focused LLM rules saved.")

In [ ]:
rule_mapping = [
    {
        "LLM_Rule_ID": "M1_M3-5.10i",
        "Part_M_Reference": "5.10i",
        "Requirement": "Minimum dimensions and arrangement comply with Diagram 18",
        "Implemented_Check": "1500mm turning circle possible from IFC bounding box",
        "IFC_Source": "IfcSpace geometry",
        "Prototype_Result": "PASS",
        "Confidence": "Partial"
    },
    {
        "LLM_Rule_ID": "M1_M3-5.10g-1",
        "Part_M_Reference": "5.10g",
        "Requirement": "Door preferably outward opening",
        "Implemented_Check": "Door width checked only; swing direction not yet implemented",
        "IFC_Source": "IfcDoor",
        "Prototype_Result": "REVIEW",
        "Confidence": "Partial"
    },
    {
        "LLM_Rule_ID": "M1_M3-5.18b",
        "Part_M_Reference": "5.18b",
        "Requirement": "Support rails and accessible shower/changing equipment provided",
        "Implemented_Check": "Doc M shower pack / support equipment detected in DDA Shower Room",
        "IFC_Source": "Contained IFC elements",
        "Prototype_Result": "PASS",
        "Confidence": "Medium"
    },
    {
        "LLM_Rule_ID": "M1_M3-5.18g",
        "Part_M_Reference": "5.18g",
        "Requirement": "Facilities for limb storage included",
        "Implemented_Check": "Limb storage object detected in DDA Shower Room",
        "IFC_Source": "Contained IFC elements",
        "Prototype_Result": "PASS",
        "Confidence": "Medium"
    },
    {
        "LLM_Rule_ID": "M1_M3-5.18j",
        "Part_M_Reference": "5.18j",
        "Requirement": "1500mm manoeuvring space provided",
        "Implemented_Check": "Minimum room plan dimension checked against 1.5m",
        "IFC_Source": "IfcSpace geometry",
        "Prototype_Result": "PASS",
        "Confidence": "Partial"
    }
]

rule_mapping_df = pd.DataFrame(rule_mapping)

rule_mapping_df

In [ ]:
rule_mapping_df.to_excel("../outputs/llm_rule_to_ifc_check_mapping.xlsx", index=False)

print("Rule mapping saved.")

In [ ]:
# Results Summary

## IFC Analysis

- 55 doors identified
- 53 spaces identified
- DDA WC detected
- DDA Shower Room detected

## Automated Compliance Checks

- Accessible WC: PASS
- Accessible Shower Room: PASS
- Door Widths: PASS
- Accessibility Fixtures: PASS
- Doc M Toilet/Shower Packs: PASS
- 1500mm Turning Space: PASS

## LLM Rule Extraction

Gemini 2.5 Flash extracted structured compliance rules from Approved Document M Section 5.

34 rules were identified covering:
- Wheelchair-accessible unisex toilets
- Wheelchair-accessible shower facilities
- Accessibility fixtures
- Manoeuvring space
- Alarm systems
- Transfer arrangements

## Rule Mapping

LLM rules were mapped to IFC entities and automated checks.

Successfully demonstrated:
- PDF → LLM Rules
- LLM Rules → IFC Entities
- IFC Entities → Compliance Results

## Limitations

The prototype currently performs object detection and geometric checks.

Detailed checks requiring:
- exact fixture coordinates
- transfer spaces
- fitting heights
- alarm reachability

remain future work.

In [ ]:
project_evidence = [
    {
        "Stage": "Part M PDF reading",
        "Output_File": "partm_accessible_wc_shower_text.txt",
        "What_It_Proves": "The system can extract relevant regulatory text from Approved Document M."
    },
    {
        "Stage": "LLM rule extraction",
        "Output_File": "focused_llm_partm_rules_wc_shower.xlsx",
        "What_It_Proves": "Gemini extracted machine-readable accessibility rules from Part M."
    },
    {
        "Stage": "IFC model extraction",
        "Output_File": "door_width_check.xlsx / dda_contained_elements.xlsx",
        "What_It_Proves": "The system can extract doors, spaces and contained accessibility fixtures from IFC."
    },
    {
        "Stage": "Geometry checking",
        "Output_File": "dda_space_dimension_checks.xlsx",
        "What_It_Proves": "The system can calculate approximate DDA WC and shower room dimensions from IFC geometry."
    },
    {
        "Stage": "Automated compliance checking",
        "Output_File": "automated_compliance_results.xlsx",
        "What_It_Proves": "Python can evaluate selected accessibility checks and generate PASS/FAIL results."
    },
    {
        "Stage": "LLM-to-IFC rule mapping",
        "Output_File": "llm_rule_to_ifc_check_mapping.xlsx",
        "What_It_Proves": "Selected LLM-extracted Part M rules were mapped to IFC data sources and prototype compliance results."
    }
]

project_evidence_df = pd.DataFrame(project_evidence)
project_evidence_df

In [ ]:
project_evidence_df.to_excel("../outputs/project_evidence_summary.xlsx", index=False)

In [ ]:
validation_rows = []

manual_keywords = {
    "Accessible WC exists": ["wheelchair-accessible unisex toilet", "provided"],
    "Toilet pack": ["wc", "toilet", "pan"],
    "Grab rails": ["grab", "rail", "support rail"],
    "Door": ["door"],
    "Transfer space": ["transfer space", "manoeuvring", "1500"],
    "Room dimensions": ["dimension", "diagram 18", "arrangement"]
}

for _, llm_rule in focused_llm_rules_df.iterrows():
    requirement = llm_rule["requirement"].lower()
    
    matched_manual_topics = []
    
    for topic, keywords in manual_keywords.items():
        if any(keyword in requirement for keyword in keywords):
            matched_manual_topics.append(topic)
    
    validation_rows.append({
        "LLM_Rule_ID": llm_rule["rule_id"],
        "Part_M_Reference": llm_rule["part_m_reference"],
        "LLM_Requirement": llm_rule["requirement"],
        "Matched_Manual_Topics": ", ".join(matched_manual_topics) if matched_manual_topics else "No direct match",
        "Validation_Status": "MATCH" if matched_manual_topics else "REVIEW"
    })

llm_validation_df = pd.DataFrame(validation_rows)

llm_validation_df

In [ ]:
# Add this cell immediately after your existing validation code

# Summary statistics
total_llm = len(llm_validation_df)
matched = len(llm_validation_df[llm_validation_df["Validation_Status"] == "MATCH"])
review = len(llm_validation_df[llm_validation_df["Validation_Status"] == "REVIEW"])

print(f"Total LLM rules extracted:     {total_llm}")
print(f"Matched to manual topics:      {matched} ({round(matched/total_llm*100)}%)")
print(f"New rules (no manual match):   {review} ({round(review/total_llm*100)}%)")
print()
print("Coverage by manual topic:")
print(llm_validation_df["Matched_Manual_Topics"].value_counts())

# Save
llm_validation_df.to_excel("../outputs/llm_vs_manual_validation.xlsx", index=False)
print("\nSaved.")

In [ ]:
llm_validation_df[
    llm_validation_df["Validation_Status"] == "REVIEW"
][[
    "LLM_Rule_ID",
    "Part_M_Reference",
    "LLM_Requirement"
]]